<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de Lenguaje Natural
# RAGs

Haremos las pruebas en el siguiente PDF: https://drive.google.com/file/d/1YcXDRlM2hY5cWHYMeyU16uruA8NgWyhl/view?usp=sharing

Instalamos librerías necesarias:
* faiss-cpu: búsqueda eficiente de vectores (similaridad)
* pypdf: para leer PDFs

In [ ]:
!pip install -q faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 27.2 MB/s eta 0:00:00


Importamos las librerias

In [ ]:
import faiss  # motor de búsqueda vectorial
import numpy as np  # operaciones numéricas
import torch  # manejo de tensores y GPU
from sentence_transformers import SentenceTransformer  # embeddings semánticos
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM  # LLM tipo encoder-decoder
from pypdf import PdfReader  # lectura de PDF

Definimos la ruta al archivo PDF

In [ ]:
pdf_path = "/content/office_rules.pdf"

Creamos un lector de PDF

In [ ]:
reader = PdfReader(pdf_path)

Definimos la variable donde acumularemos todo el texto

In [ ]:
raw_text = ""

Iteramos página por página

In [ ]:
for page in reader.pages:
    text = page.extract_text()  # extrae texto de cada página
    if text:  # evitamos páginas vacías
        raw_text += text + "\n"  # concatenamos con salto de línea

Mostramos tamaño total del texto

In [ ]:
print("Longitud texto:", len(raw_text))

Longitud texto: 1112


Mostramos el texto

In [ ]:
print(raw_text)

Office Rules (Inspired by The Office)
1. Meetings should have a clear purpose. If the meeting could have been an email, it probably should
have been.
2. Respect your coworkers’ time. Avoid unnecessary interruptions unless it's truly important.
3. Keep humor light and inclusive. A good joke should make everyone comfortable.
4. Stay organized. A clean desk helps maintain a clear mind.
5. Celebrate small wins. Recognition builds a positive work environment.

Workplace Culture Guidelines
6. Communicate clearly and respectfully. Misunderstandings slow everyone down.
7. Take responsibility for your work. Accountability builds trust.
8. Help your team when possible. Collaboration is key to success.
9. Keep learning. Growth is part of every role.
10. Stay professional, even in stressful situations.

Fun but Professional Rules
11. Friendly competition is welcome, but teamwork comes first.
12. Personal quirks are okay—just be mindful of others.
13. Breaks are important. Step away and recharge wh

Definimos la función de chunk, es similar al anterior pero usa caracteres en lugar de palabras, para más estabilidad

In [ ]:
def chunk_text(text, chunk_size=800, overlap=150):
    """
    Divide el texto en fragmentos (chunks).

    - chunk_size: tamaño máximo del chunk en caracteres
    - overlap: solapamiento entre chunks (importante para no perder contexto)

    Usar caracteres en lugar de palabras es más estable para PDFs.
    """
    chunks = []

    # Iteramos en pasos de (chunk_size - overlap)
    for i in range(0, len(text), chunk_size - overlap):
        chunk = text[i:i + chunk_size]  # cortamos fragmento

        # Filtramos chunks muy pequeños (ruido)
        if len(chunk.strip()) > 50:
            chunks.append(chunk)

    return chunks

Generamos los chunks del documento

In [ ]:
chunks = chunk_text(raw_text)

Vemos cuántos fragmentos tenemos

In [ ]:
print("Total chunks:", len(chunks))

Total chunks: 2


### **1) Embeddings**

Cargamos modelo de embeddings. Ahora usamos "bge-small-en", es más potente que MiniLM en muchos casos

In [ ]:
embedder = SentenceTransformer("BAAI/bge-small-en")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Convertimos cada chunk a un vector (embedding)

In [ ]:
embeddings = embedder.encode(
    chunks,
    convert_to_numpy=True,      # salida como numpy array (para FAISS)
    normalize_embeddings=True   # permite usar cosine similarity
)

Vemos la dimensión de los embeddings

In [ ]:
dimension = embeddings.shape[1]

IndexFlatIP es para producto interno. Si los vectores están normalizados esto equivaldria a cosine similarity

In [ ]:
index = faiss.IndexFlatIP(dimension)

Agregamos todos los embeddings al índice

In [ ]:
index.add(embeddings)

Definimos el mapeo: índice a texto original

In [ ]:
id_to_doc = {i: doc for i, doc in enumerate(chunks)}

Definimos la función mejorada

In [ ]:
def search_top_chunks(query, k=3):
    """
    Dada una pregunta:
    - la convierte en embedding
    - busca los k chunks más similares
    """

    # Convertimos query a vector
    query_vector = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # FAISS busca los k más cercanos
    D, I = index.search(query_vector, k)

    # I contiene índices, D contiene similitudes

    # Eliminamos índices inválidos (-1)
    valid_indices = [i for i in I[0] if i != -1]

    # Retornamos los textos correspondientes
    return [id_to_doc[i] for i in valid_indices]


### **2) Large Language Model**

Detectamos si hay GPU disponible

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

Modelo generativo tipo instrucción

In [ ]:
model_name = "google/flan-t5-large"  # bueno para QA

Definimos tokenizador

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Definimos el modelo

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Definimos la función de generación

In [ ]:
def generate(prompt, max_new_tokens=200):
    """
    Genera texto a partir de un prompt.
    """

    # Tokenizamos entrada
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,  # evita textos demasiado largos
        max_length=1024   # límite del modelo
    ).to(device)

    # Generación determinista (sin sampling)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,   # salida más estable
        temperature=0.0    # completamente determinista
    )

    # Convertimos tokens a texto
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

### **3) Chatbot RAG**

Definimos la función de chatbot con prompt que incluye contexto y reglas en ingles

In [ ]:
def chatbot_ingles(query):
    """
    Pipeline completo RAG:
    1. Busca contexto relevante
    2. Construye prompt
    3. Genera respuesta
    """

    # Recuperamos los chunks más relevantes
    top_chunks = search_top_chunks(query, k=3)

    # Limitamos el contexto (evita overflow y ruido)
    context = "\n\n".join(top_chunks[:3])

    # Prompt tipo instruction-following
    prompt = f"""
You are a helpful assistant.

Answer the question using ONLY the information from the context below.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question:
{query}

Answer:
"""

    # Generamos respuesta
    return generate(prompt)

Realizamos pruebas con el chatbot con prompt con contexto y reglas básicas en ingles

In [ ]:
print("Q:", "What is this document about?")
print("A:", chatbot_ingles("What is this document about?"))

Q: What is this document about?


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


A: Office Rules


In [ ]:
print("Q:", "What is the third rule?")
print("A:", chatbot_ingles("What is the third rule?"))

Q: What is the third rule?
A: Keep humor light and inclusive


In [ ]:
print("Q:", "Which rule talks about 'clean desk helps maintain a clear mind'?")
print("A:", chatbot_ingles("Which rule talks about 'clean desk helps maintain a clear mind'?"))

Q: Which rule talks about 'clean desk helps maintain a clear mind'?
A: 4.
